<a href="https://colab.research.google.com/github/slomi23/NLP_final_project/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup
Mount Google Drive and clone the project repo. Run this first in every new session.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!rm -rf NLP_final_project
!git clone https://github.com/slomi23/NLP_final_project.git
%cd /content/NLP_final_project

/content
Cloning into 'NLP_final_project'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 247 (delta 26), reused 32 (delta 9), pack-reused 174 (from 2)
Receiving objects: 100% (247/247), 151.38 MiB | 30.87 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Filtering content: 100% (15/15), 2.04 GiB | 14.23 MiB/s, done.
/content/NLP_final_project


## Chunk Quality Check
Scan chunks.jsonl for low quality chunks, specifically lines with excessive dot sequences. Prints a count and previews the worst offenders.

In [4]:
import json, re

path = "/content/NLP_final_project/data/jurafsky_chunks/chunks.jsonl"
bad = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        text = c["text"]
        if re.search(r"(?:\.\s*){5,}", text) or " . . . " in text:
            bad.append(text[:1000])

print("Bad dot chunks:", len(bad))
for x in bad[:3]:
    print("=" * 80)
    print(x)

Bad dot chunks: 71
Pty. Limited, Sydney Prentice-Hall Canada, Inc., Toronto Prentice-Hall Hispanoamericana, S.A., Mexico Prentice-Hall of India Private Limited, New Delhi Prentice-Hall of Japan, Inc., Tokyo Simon & Schuster Asia Pte. Ltd., Singapore Editora Prentice-Hall do Brasil, Ltda., Rio de Janeiro For my parents— D.J. For Linda — J.M. Summary of Contents 1 Introduction. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .. . . . . . 1 I Words 19 2 Regular Expressions and Automata. . . . . . . . . . . . . . . . . . . . .. 21 3 Morphology and Finite-State Transducers . . . . . . . . . . . . . .. 57 4 Computational Phonology and Text-to-Speech . . . . . . . . . . . 91 5 Probabilistic Models of Pronunciation and Spelling . . . . . . 139 6 N-grams . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .. . . . 189 7 HMMs and Speech Recognition . . . . . . . . . . . . . . . . . . . . . .
. . . 233 II Syntax 283 8 Word Classes and Part-o

## Import and Reload Model Modules
Add the project root to `sys.path` and force-reload `encoder` and `loss` modules
so that any edits made during development are picked up without restarting the runtime.



In [5]:
import sys
import importlib
from pathlib import Path

project_root = Path("/content/NLP_final_project").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if not project_root.exists():
    raise FileNotFoundError("Repo not found. Run the clone cell first.")

import src.models.encoder as encoder_module
import src.models.loss as loss_module

importlib.reload(encoder_module)
importlib.reload(loss_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer
InfoNCELoss = loss_module.InfoNCELoss

print("Loading Done")

Project root: /content/NLP_final_project
Project exists: True
✅ Reloaded fixed encoder/loss
Files in project root:
 - encoder_transformer.pth
 - .gitattributes
 - data
 - models
 - notebooks
 - chunk_tfidf_analysis.json
 - src
 - report
 - .idea
 - README.md
 - .git
 - training.ipynb


## Rebuild Jurafsky Chunks
Installs `pypdf`, verifies the Jurafsky PDF exists, then re-runs `book_chunks.py`
to regenerate `chunks.jsonl` from scratch. Run this whenever the chunking logic changes.

In [6]:
%cd /content/NLP_final_project
!pip -q install pypdf

PDF_PATH = project_root / "data" / "raw" / "Speech_and_Language_Processing.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Missing Jurafsky PDF: {PDF_PATH}")

!rm -f data/jurafsky_chunks/chunks.jsonl
!python src/data/book_chunks.py


/content/NLP_final_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 10.4 MB/s eta 0:00:00
PDF exists: True /content/NLP_final_project/data/raw/Speech_and_Language_Processing.pdf
CURRENT DIR: /content/NLP_final_project
Extracting text from PDF: data/raw/Speech_and_Language_Processing.pdf
Cleaning text...
Splitting into paragraphs...
Found 590 useful paragraphs
Building chunks...
Created 663 clean chunks
Saved to data/jurafsky_chunks/chunks.jsonl

Quality report:
  chunks: 663
  dot-leader chunks: 22
  numeric-heavy chunks: 0

Example bad dot-leader chunk:
Section 2.1. Regular Expressions 25 RE Match (single characters) Example Patterns Matched [ˆA-Z] not an uppercase letter “Oyfn pripetchik” [ˆSs] neither ‘S’ nor ‘s’ “I have no exquisite reason for’t” [ˆ\.] not a period “our resident Djinn” [eˆ] either ‘e’ or ‘ˆ’ “look up ˆ now” aˆb the pattern ‘aˆb’ “look up aˆ bnow” Figure 2.3 Uses of the caretˆ for negation or just to meanˆ RE Match Example Patterns Matched woodchuc

## Load & Inspect Jurafsky Chunks
Loads all chunks into memory and prints word-count statistics (min/mean/max).
Previews the first 3 chunks to sanity-check content quality.

In [7]:
import json
import numpy as np

CHUNKS_FILE = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"

if not CHUNKS_FILE.exists():
    raise FileNotFoundError(CHUNKS_FILE)

chunks = []
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f" Loaded {len(chunks)} clean Jurafsky chunks")
word_counts = [len(c["text"].split()) for c in chunks]
print(f"Word count min/mean/max: {min(word_counts)} / {np.mean(word_counts):.1f} / {max(word_counts)}")

for i, c in enumerate(chunks[:3]):
    print("\n" + "=" * 90)
    print("CHUNK", i, "word_count:", len(c["text"].split()))
    print(c["text"][:1200])


✅ Loaded 663 clean Jurafsky chunks
Word count min/mean/max: 180 / 246.0 / 448

CHUNK 0 word_count: 250
Preface This is an exciting time to be working in speech and language p rocessing. Historically distinct ﬁelds (natural language processing, speech recognition, computational linguistics, computational psycholinguis tics) have begun to merge. The commercial availability of speech recognition, and the need for web-based language techniques have provided an important impetus for development of real systems. The availability of very large on-line corpora has enabled statistical models of language at every level, f rom phonetics to discourse. We have tried to draw on this emerging state of the art in the design of this pedagogical and reference work: 1. Coverage In attempting to describe a uniﬁed vision of speech and langu age pro- cessing, we cover areas that traditionally are taught in different courses in different departments: speech recognition in electrica l engineering, parsing, se

##Triplet Generation 1
An earlier, simpler version of triplet construction: loads MS MARCO, ArXiv, and
Jurafsky data, adds a single random negative per triplet, and does a 90/10 split.
**Kept for reference only — the refined v2 pipeline below replaces this.**

In [8]:
import json
import random
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

QUERIES_FILE = project_root / "data" / "processed" / "msmarco_small_queries.json"
PASSAGES_FILE = project_root / "data" / "processed" / "msmarco_small_passages.json"
QRELS_FILE = project_root / "data" / "processed" / "msmarco_small_qrels.json"

with open(QUERIES_FILE, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

with open(PASSAGES_FILE, 'r', encoding='utf-8') as f:
    passages_list = json.load(f)

with open(QRELS_FILE, 'r', encoding='utf-8') as f:
    qrels = json.load(f)

query_lookup = {q['id']: q['text'] for q in queries_list}
passage_lookup = {p['id']: p['text'] for p in passages_list}

ms_marco_triplets = []

for query_id, relevant_passages in qrels.items():
    if query_id not in query_lookup:
        continue

    query_text = query_lookup[query_id]

    best_pid, _ = max(relevant_passages, key=lambda x: x)

    if best_pid not in passage_lookup:
        continue

    positive_text = passage_lookup[best_pid]

    if len(query_text.split()) < 3:
        continue

    ms_marco_triplets.append({
        'query': query_text,
        'positive': positive_text,
        'negatives': []
    })

ARXIV_TRAIN_FILE = project_root / "data" / "processed" / "arxiv_train_triplets.jsonl"
arxiv_triplets = []
with open(ARXIV_TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            arxiv_triplets.append(json.loads(line.strip()))

chunks_file = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"
chunks = []
with open(chunks_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line.strip()))

def extract_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    valid_sentences = [
        s.strip() for s in sentences
        if len(s.split()) > 5 and not s.startswith('(') and not s.startswith('[')
    ]
    return valid_sentences

jurafsky_triplets = []
for chunk in chunks:
    text = chunk['text']
    sentences = extract_sentences(text)

    for sentence in sentences:
        if len(sentence.split()) >= 10:
            jurafsky_triplets.append({
                'query': sentence,
                'positive': text,
                'negatives': [] # Will add negatives later
            })

arxiv_file = project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv"
arxiv_df = pd.read_csv(arxiv_file)
arxiv_df['combined_text'] = arxiv_df.apply(
    lambda row: f"Title: {row['title']}. Abstract: {row['abstract']}" if pd.notna(row['title']) and pd.notna(row['abstract']) else "",
    axis=1
)

arxiv_pool = arxiv_df['combined_text'].tolist()
jurafsky_pool = [chunk['text'] for chunk in chunks]
negative_pool = arxiv_pool + jurafsky_pool

def add_negatives(triplets, neg_pool, n_negatives=1):
    for triplet in triplets:
        negs = []
        while len(negs) < n_negatives:
            neg = random.choice(neg_pool)
            if neg != triplet['positive']:
                negs.append(neg)
        triplet['negatives'] = negs
    return triplets

ms_marco_triplets = add_negatives(ms_marco_triplets, negative_pool)
jurafsky_triplets = add_negatives(jurafsky_triplets, negative_pool)
if arxiv_triplets and 'negatives' not in arxiv_triplets:
    arxiv_triplets = add_negatives(arxiv_triplets, negative_pool)

all_triplets = arxiv_triplets + ms_marco_triplets + jurafsky_triplets

train_triplets, val_triplets = train_test_split(
    all_triplets,
    test_size=0.1,
    random_state=42
)

print(f"\n Final Split:")
print(f"   - Training Triplets: {len(train_triplets)}")
print(f"   - Validation Triplets: {len(val_triplets)}")

OUT_COMBINED_TRAIN = project_root / "data" / "processed" / "combined_train_triplets.jsonl"
OUT_COMBINED_VAL   = project_root / "data" / "processed" / "combined_val_triplets.jsonl"

def write_triplets(triplets, path):
    with open(path, 'w', encoding='utf-8') as f:
        for t in triplets:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"  wrote {len(triplets):,} triplets → {path.name}")

write_triplets(train_triplets, OUT_COMBINED_TRAIN)
write_triplets(val_triplets, OUT_COMBINED_VAL)


📚 Loading MS MARCO data directly from JSON files...
✅ Loaded 20000 queries
✅ Loaded 10000 passages
✅ Loaded 20000 relevance judgments
🔄 Generating MS MARCO triplets...
✅ Generated 20000 MS MARCO pairs
📚 Loading ArXiv triplets...
✅ Loaded 13500 ArXiv triplets
📚 Generating Jurafsky Book triplets...
✅ Generated 5979 Jurafsky triplets
🔄 Preparing negative pool...
✅ Negative pool size: 417994 passages
🔄 Adding negatives to all triplets...
✅ Added negatives to all triplets

📊 Total Combined Triplets: 39479
   - ArXiv: 13500
   - MS MARCO: 20000
   - Jurafsky: 5979

📂 Final Split:
   - Training Triplets: 35531
   - Validation Triplets: 3948
  wrote 35,531 triplets → combined_train_triplets.jsonl
  wrote 3,948 triplets → combined_val_triplets.jsonl
✅ Data preparation complete! Ready for training.


## Locate ArXiv CSV
Searches both the repo and Google Drive for `arxiv_cs_papers_processed.csv`.
Prints a quick 5-row preview to confirm the file is readable before the expensive
triplet generation step.

In [9]:
import pandas as pd
import numpy as np

ARXIV_CANDIDATES = [
    project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv",
    Path("/content/drive/MyDrive/NLP_project/neural_search_real_pipeline_restored/data/processed/arxiv_cs_papers_processed.csv"),
]

ARXIV_FILE = None
for p in ARXIV_CANDIDATES:
    if p.exists():
        ARXIV_FILE = p
        break

if ARXIV_FILE is None:
    raise FileNotFoundError("Could not find arxiv_cs_papers_processed.csv in repo or Drive path.")

sample_arxiv = pd.read_csv(ARXIV_FILE, usecols=["title", "abstract"], nrows=5)


ArXiv file: /content/NLP_final_project/data/processed/arxiv_cs_papers_processed.csv
                                               title  \
0  The Impact of Socioeconomic Factors on Health ...   
1  Five Properties of Specific Curiosity You Didn...   
2  Shape-Guided Diffusion with Inside-Outside Att...   
3  Test-Time Mixup Augmentation for Data and Clas...   
4  Modeling Complex Dialogue Mappings via Sentenc...   

                                            abstract  
0  High-quality healthcare in the US can be cost-...  
1  Curiosity for machine agents has been a focus ...  
2  We introduce precise object silhouette as a ne...  
3  Uncertainty estimation of trained deep learnin...  
4  Complex dialogue mappings (CDM), including one...  


##Triplet Generation v2 — Multi-Source with Hard Negatives
Builds training triplets from three sources with capped sizes for speed:
- **ArXiv** (up to 40k): title → full title+abstract pairs
- **MS MARCO** (up to 5k): query → best relevant passage pairs
- **Jurafsky** (synthetic): sentence → parent chunk pairs

Each triplet gets **4 negatives** drawn from a mixed-domain pool (ArXiv + Jurafsky +
MS MARCO), with at least 1 same-domain negative per triplet.

In [10]:
import json
import random
import re
from pathlib import Path
import pandas as pd

random.seed(42)

MAX_ARXIV_TRIPLETS = 40000
MAX_MSMARCO_TRIPLETS = 5000
N_NEGATIVES = 4


QUERIES_FILE = project_root / "data" / "processed" / "msmarco_small_queries.json"
PASSAGES_FILE = project_root / "data" / "processed" / "msmarco_small_passages.json"
QRELS_FILE = project_root / "data" / "processed" / "msmarco_small_qrels.json"

for path in [QUERIES_FILE, PASSAGES_FILE, QRELS_FILE]:
    print(path.name, "exists:", path.exists())

ms_marco_triplets = []
passages_list = []

if QUERIES_FILE.exists() and PASSAGES_FILE.exists() and QRELS_FILE.exists():
    with open(QUERIES_FILE, "r", encoding="utf-8") as f:
        queries_list = json.load(f)

    with open(PASSAGES_FILE, "r", encoding="utf-8") as f:
        passages_list = json.load(f)

    with open(QRELS_FILE, "r", encoding="utf-8") as f:
        qrels = json.load(f)

    query_lookup = {str(q["id"]): q["text"] for q in queries_list}
    passage_lookup = {str(p["id"]): p["text"] for p in passages_list}

    for query_id, relevant_passages in qrels.items():
        query_id = str(query_id)

        if query_id not in query_lookup or not relevant_passages:
            continue

        query_text = query_lookup[query_id].strip()

        best_pid, best_score = max(relevant_passages, key=lambda x: x[1])
        best_pid = str(best_pid)

        if best_pid not in passage_lookup:
            continue

        positive_text = passage_lookup[best_pid].strip()

        if len(query_text.split()) < 3 or len(positive_text.split()) < 10:
            continue

        ms_marco_triplets.append({
            "query": query_text,
            "positive": positive_text,
            "negatives": [],
            "source": "msmarco",
        })

    random.shuffle(ms_marco_triplets)
    ms_marco_triplets = ms_marco_triplets[:MAX_MSMARCO_TRIPLETS]

arxiv_df = pd.read_csv(ARXIV_FILE, usecols=["title", "abstract"])
arxiv_df = arxiv_df.dropna(subset=["title", "abstract"])

arxiv_df["title"] = arxiv_df["title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
arxiv_df["abstract"] = arxiv_df["abstract"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

arxiv_df = arxiv_df[
    (arxiv_df["title"].str.len() > 10) &
    (arxiv_df["abstract"].str.len() > 120)
]

if len(arxiv_df) > MAX_ARXIV_TRIPLETS:
    arxiv_df = arxiv_df.sample(n=MAX_ARXIV_TRIPLETS, random_state=42)

arxiv_df["combined_text"] = (
    "Title: " + arxiv_df["title"] +
    ". Abstract: " + arxiv_df["abstract"]
)

arxiv_texts = arxiv_df["combined_text"].tolist()
arxiv_titles = arxiv_df["title"].tolist()

arxiv_triplets = [
    {
        "query": title,
        "positive": full_text,
        "negatives": [],
        "source": "arxiv",
    }
    for title, full_text in zip(arxiv_titles, arxiv_texts)
]

def extract_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [
        s.strip()
        for s in sentences
        if len(s.split()) >= 8
        and len(s.split()) <= 40
        and not s.strip().startswith("(")
        and not s.strip().startswith("[")
    ]

jurafsky_triplets = []

for chunk in chunks:
    text = chunk.get("text", "")

    if not isinstance(text, str) or not text.strip():
        continue

    sentences = extract_sentences(text)

    if len(sentences) > 3:
        sentences = random.sample(sentences, 3)

    first_words = " ".join(text.split()[:18])
    candidate_queries = sentences + [first_words]

    for q in candidate_queries:
        jurafsky_triplets.append({
            "query": q,
            "positive": text,
            "negatives": [],
            "source": "jurafsky",
        })

jurafsky_pool = [
    c["text"]
    for c in chunks
    if isinstance(c.get("text", ""), str) and len(c["text"].split()) > 80
]

msmarco_pool = [
    p["text"]
    for p in passages_list
    if isinstance(p.get("text", ""), str) and len(p["text"].split()) > 20
]

arxiv_pool = arxiv_texts

all_negative_pool = list(dict.fromkeys(arxiv_pool + jurafsky_pool + msmarco_pool))

def normalize_negatives(negs):
    if isinstance(negs, str):
        return [negs]
    if isinstance(negs, list):
        return [n for n in negs if isinstance(n, str) and n.strip()]
    return []

def add_negatives(triplets, same_domain_pool, all_pool, n_negatives=4):
    fixed = []

    for triplet in triplets:
        query = triplet.get("query", "")
        positive = triplet.get("positive", "")

        if not isinstance(query, str) or not isinstance(positive, str):
            continue

        query = query.strip()
        positive = positive.strip()

        if not query or not positive:
            continue

        existing = normalize_negatives(triplet.get("negatives", []))

        tries = 0
        while len(existing) < 1 and same_domain_pool and tries < 100:
            neg = random.choice(same_domain_pool)
            tries += 1
            if neg != positive and neg not in existing:
                existing.append(neg)

        tries = 0
        while len(existing) < n_negatives and tries < 1300:
            neg = random.choice(all_pool)
            tries += 1
            if neg != positive and neg not in existing:
                existing.append(neg)

        if len(existing) >= n_negatives:
            new_t = dict(triplet)
            new_t["query"] = query
            new_t["positive"] = positive
            new_t["negatives"] = existing[:n_negatives]
            fixed.append(new_t)

    return fixed

arxiv_triplets = add_negatives(arxiv_triplets, arxiv_pool, all_negative_pool, N_NEGATIVES)
ms_marco_triplets = add_negatives(ms_marco_triplets, msmarco_pool, all_negative_pool, N_NEGATIVES)
jurafsky_triplets = add_negatives(jurafsky_triplets, jurafsky_pool, all_negative_pool, N_NEGATIVES)

all_triplets_raw = arxiv_triplets + ms_marco_triplets + jurafsky_triplets

bad = [
    t for t in all_triplets_raw
    if "negatives" not in t
    or not isinstance(t["negatives"], list)
    or len(t["negatives"]) < N_NEGATIVES
]

if bad:
    raise ValueError("Some triplets still have missing/bad negatives.")

OUT_ALL_TRIPLETS = project_root / "data" / "processed" / "combined_all_triplets.jsonl"
with open(OUT_ALL_TRIPLETS, "w", encoding="utf-8") as f:
    for t in all_triplets_raw:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")


📚 Loading MS MARCO data directly from JSON files...
msmarco_small_queries.json exists: True
msmarco_small_passages.json exists: True
msmarco_small_qrels.json exists: True
⚠️ MS MARCO file looks synthetic. Using it lightly only.
✅ MS MARCO triplets used: 5000
📚 Loading ArXiv CSV and creating ArXiv title→abstract triplets...
✅ ArXiv triplets used: 40000
📚 Generating Jurafsky sentence→chunk triplets...
✅ Jurafsky triplets generated: 2635
🔄 Preparing negative pools...
Negative pools:
   - ArXiv:    40000
   - Jurafsky: 663
   - MS MARCO: 10000
   - All:      39322
🔄 Adding negatives...

📊 Triplets before split:
   - ArXiv:     40000
   - MS MARCO:  5000
   - Jurafsky:  2635
   - TOTAL:     47635
Bad triplets without 4 negatives: 0
✅ Saved all raw triplets to: /content/NLP_final_project/data/processed/combined_all_triplets.jsonl


##Deduplicate, Split & Balance
Removes duplicate triplets, does a per-source 95/5 train/val split to prevent
data leakage, then **5× oversamples Jurafsky** in the training set only — boosting
NLP textbook retrieval performance without contaminating validation metrics.
Saves final splits to `combined_train_triplets.jsonl` and `combined_val_triplets.jsonl`.

In [11]:
import random
import json
from collections import Counter
from sklearn.model_selection import train_test_split


N_NEGATIVES = 4
JURAFSKY_TRAIN_MULTIPLIER = 5

def clean_one(t):
    query = t.get("query", "")
    positive = t.get("positive", "")
    negatives = t.get("negatives", [])
    source = t.get("source", "unknown")

    if not isinstance(query, str) or not isinstance(positive, str):
        return None
    if not isinstance(negatives, list):
        return None

    query = query.strip()
    positive = positive.strip()

    negatives = [
        n.strip()
        for n in negatives
        if isinstance(n, str) and n.strip() and n.strip() != positive
    ]

    if not query or not positive or len(negatives) < N_NEGATIVES:
        return None

    return {
        "query": query,
        "positive": positive,
        "negatives": negatives,
        "source": source,
    }

clean_triplets = []
seen = set()

for t in all_triplets_raw:
    ct = clean_one(t)
    if ct is None:
        continue

    key = (ct["query"], ct["positive"], ct["source"])
    if key in seen:
        continue

    seen.add(key)
    clean_triplets.append(ct)

by_source = {}
for t in clean_triplets:
    by_source.setdefault(t["source"], []).append(t)

train_triplets_base = []
val_triplets = []

for source, items in by_source.items():
    random.seed(42)
    random.shuffle(items)

    if len(items) < 10:
        train_items = items
        val_items = []
    else:
        train_items, val_items = train_test_split(
            items,
            test_size=0.05,
            random_state=42
        )

    train_triplets_base.extend(train_items)
    val_triplets.extend(val_items)

jur_train = [t for t in train_triplets_base if t["source"] == "jurafsky"]
non_jur_train = [t for t in train_triplets_base if t["source"] != "jurafsky"]

train_triplets = non_jur_train + (jur_train * JURAFSKY_TRAIN_MULTIPLIER)

random.seed(42)
random.shuffle(train_triplets)
random.shuffle(val_triplets)

print("Train source counts:", Counter(t["source"] for t in train_triplets))
print("Val source counts:", Counter(t["source"] for t in val_triplets))

OUT_COMBINED_TRAIN = project_root / "data" / "processed" / "combined_train_triplets.jsonl"
OUT_COMBINED_VAL = project_root / "data" / "processed" / "combined_val_triplets.jsonl"

def write_triplets(triplets, path):
    with open(path, "w", encoding="utf-8") as f:
        for t in triplets:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"  wrote {len(triplets):,} triplets → {path}")

write_triplets(train_triplets, OUT_COMBINED_TRAIN)
write_triplets(val_triplets, OUT_COMBINED_VAL)



🔄 Deduplicating, splitting and balancing triplets...
Clean unique triplets: 41464
Source counts: Counter({'arxiv': 38640, 'jurafsky': 2634, 'msmarco': 190})

📂 Final Split:
   - Base train before oversampling: 39390
   - Training Triplets:             49398
   - Validation Triplets:           2074
Train source counts: Counter({'arxiv': 36708, 'jurafsky': 12510, 'msmarco': 180})
Val source counts: Counter({'arxiv': 1932, 'jurafsky': 132, 'msmarco': 10})
  wrote 49,398 triplets → /content/NLP_final_project/data/processed/combined_train_triplets.jsonl
  wrote 2,074 triplets → /content/NLP_final_project/data/processed/combined_val_triplets.jsonl
✅ Data split complete.


##Training — InfoNCE Contrastive Loss
Trains the encoder for 4 epochs using **InfoNCE loss** (temperature=0.07) with
AdamW (lr=1e-4, weight_decay=1e-4). Each batch encodes:
- 1 query embedding
- 1 positive embedding
- 4 negative embeddings (reshaped to `[batch, 4, d_model]`)

Validation loss is computed after each epoch. Model and tokenizer are saved locally
and backed up to Google Drive to survive session resets.

In [16]:
import gc
torch.cuda.empty_cache()
gc.collect()

import torch
import torch.optim as optim
import random
import numpy as np
from pathlib import Path
import sys
import time
import json
import importlib
import shutil

import src.models.encoder as encoder_module
import src.models.loss as loss_module

importlib.reload(encoder_module)
importlib.reload(loss_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer
InfoNCELoss = loss_module.InfoNCELoss

CONFIG = {
    "vocab_size": 50000,
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 3,
    "d_ff": 512,
    "max_len": 192,
    "dropout": 0.2,
    "pad_token_id": 0
}

TRAINING_CONFIG = {
    "epochs": 4,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "temperature": 0.07,
    "weight_decay": 1e-4,
    "n_negatives": 4,
    "grad_accum_steps": 4
}

MODEL_SAVE_PATH = project_root / "models" / "encoder_transformer.pth"
TOKENIZER_SAVE_PATH = project_root / "models" / "tokenizer_vocab.json"

MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Training triplets:", len(train_triplets))
print("Validation triplets:", len(val_triplets))

if len(train_triplets) == 0:
    raise ValueError("train_triplets is empty. Run triplet and split cells first.")

N_NEGATIVES = TRAINING_CONFIG["n_negatives"]

bad_train = [
    t for t in train_triplets
    if "negatives" not in t
    or not isinstance(t["negatives"], list)
    or len(t["negatives"]) < N_NEGATIVES
]

print("Bad train triplets:", len(bad_train))

if bad_train:
    raise ValueError("Some training triplets have bad negatives.")

model = EncoderOnlyTransformer(CONFIG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to_device(device)

all_texts_for_vocab = []

for t in train_triplets:
    all_texts_for_vocab.append(t["query"])
    all_texts_for_vocab.append(t["positive"])

    for neg in t["negatives"]:
        if isinstance(neg, str) and neg.strip():
            all_texts_for_vocab.append(neg)

tokenizer = SimpleTokenizer(CONFIG["vocab_size"])
tokenizer.fit(all_texts_for_vocab)

tokenizer.save(str(TOKENIZER_SAVE_PATH))

def get_negatives(t, n_negatives):
    negs = t.get("negatives", [])

    if isinstance(negs, str):
        negs = [negs]

    negs = [
        n for n in negs
        if isinstance(n, str) and n.strip()
    ]

    if len(negs) < n_negatives:
        raise ValueError("Triplet does not have enough negatives.")

    return negs[:n_negatives]

def generate_batches(triplets, tokenizer, batch_size, max_len, n_negatives, shuffle=True):
    triplets = list(triplets)

    if shuffle:
        random.shuffle(triplets)

    for i in range(0, len(triplets), batch_size):
        batch_triplets = triplets[i:i + batch_size]

        queries = [t["query"] for t in batch_triplets]
        positives = [t["positive"] for t in batch_triplets]

        negatives_nested = [
            get_negatives(t, n_negatives)
            for t in batch_triplets
        ]

        flat_negatives = [
            neg
            for negs in negatives_nested
            for neg in negs
        ]

        q_encoded = tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        p_encoded = tokenizer(
            positives,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        n_encoded = tokenizer(
            flat_negatives,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        yield {
            "query_ids": q_encoded["input_ids"],
            "query_mask": q_encoded["attention_mask"],
            "pos_ids": p_encoded["input_ids"],
            "pos_mask": p_encoded["attention_mask"],
            "neg_ids": n_encoded["input_ids"],
            "neg_mask": n_encoded["attention_mask"],
            "batch_size_actual": len(batch_triplets)
        }

optimizer = optim.AdamW(
    model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"]
)

criterion = InfoNCELoss(temperature=TRAINING_CONFIG["temperature"])

num_epochs = TRAINING_CONFIG["epochs"]
batch_size = TRAINING_CONFIG["batch_size"]
max_len = CONFIG["max_len"]
n_negatives = TRAINING_CONFIG["n_negatives"]

start_time = time.time()

for epoch in range(num_epochs):
    model.train()

    epoch_loss = 0.0
    num_batches = 0

    data_gen = generate_batches(
        train_triplets,
        tokenizer,
        batch_size,
        max_len,
        n_negatives,
        shuffle=True
    )

    for batch_idx, batch in enumerate(data_gen):
        q_ids = batch["query_ids"].to(device)
        q_mask = batch["query_mask"].to(device)

        p_ids = batch["pos_ids"].to(device)
        p_mask = batch["pos_mask"].to(device)

        n_ids = batch["neg_ids"].to(device)
        n_mask = batch["neg_mask"].to(device)

        bsz = batch["batch_size_actual"]

        anchor_emb = model(q_ids, q_mask)
        pos_emb = model(p_ids, p_mask)

        neg_emb_flat = model(n_ids, n_mask)
        neg_emb = neg_emb_flat.view(bsz, n_negatives, CONFIG["d_model"])

        loss = criterion(anchor_emb, pos_emb, neg_emb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

        if batch_idx % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Batch {batch_idx} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = epoch_loss / max(num_batches, 1)
    print(f" Epoch {epoch + 1} Complete | Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    val_loss = 0.0
    val_batches = 0

    val_gen = generate_batches(
        val_triplets,
        tokenizer,
        batch_size,
        max_len,
        n_negatives,
        shuffle=False
    )

    with torch.no_grad():
        for batch in val_gen:
            v_q = batch["query_ids"].to(device)
            v_q_mask = batch["query_mask"].to(device)

            v_p = batch["pos_ids"].to(device)
            v_p_mask = batch["pos_mask"].to(device)

            v_n = batch["neg_ids"].to(device)
            v_n_mask = batch["neg_mask"].to(device)

            bsz = batch["batch_size_actual"]

            v_anchor = model(v_q, v_q_mask)
            v_pos = model(v_p, v_p_mask)

            v_neg_flat = model(v_n, v_n_mask)
            v_neg = v_neg_flat.view(bsz, n_negatives, CONFIG["d_model"])

            v_loss = criterion(v_anchor, v_pos, v_neg)

            val_loss += v_loss.item()
            val_batches += 1

    avg_val_loss = val_loss / max(val_batches, 1)
    print(f" Validation Loss: {avg_val_loss:.4f}")

total_time = time.time() - start_time

print(f"\n Training complete in {total_time / 60:.1f} minutes")

model.save(str(MODEL_SAVE_PATH))

DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/NLP_project/saved_models")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(MODEL_SAVE_PATH, DRIVE_MODEL_DIR / "encoder_transformer.pth")
shutil.copy2(TOKENIZER_SAVE_PATH, DRIVE_MODEL_DIR / "tokenizer_vocab.json")



Training triplets: 49398
Validation triplets: 2074
Bad train triplets: 0
🧠 Initializing Encoder Model...
🚀 Using device: cuda
🔄 Fitting tokenizer on train data...
✅ Vocabulary size: 50000
✅ Saved tokenizer to: /content/NLP_final_project/models/tokenizer_vocab.json
🚀 Starting Training...
Epoch 1/4 | Batch 0 | Loss: 1.7774
Epoch 1/4 | Batch 100 | Loss: 1.5543
Epoch 1/4 | Batch 200 | Loss: 1.4842
Epoch 1/4 | Batch 300 | Loss: 1.5284
Epoch 1/4 | Batch 400 | Loss: 1.3778
Epoch 1/4 | Batch 500 | Loss: 1.4833
Epoch 1/4 | Batch 600 | Loss: 1.3739
Epoch 1/4 | Batch 700 | Loss: 1.3577
Epoch 1/4 | Batch 800 | Loss: 1.3983
Epoch 1/4 | Batch 900 | Loss: 1.5148
Epoch 1/4 | Batch 1000 | Loss: 1.4394
Epoch 1/4 | Batch 1100 | Loss: 1.5098
Epoch 1/4 | Batch 1200 | Loss: 1.1289
Epoch 1/4 | Batch 1300 | Loss: 0.9763
Epoch 1/4 | Batch 1400 | Loss: 0.9246
Epoch 1/4 | Batch 1500 | Loss: 0.6442
🏁 Epoch 1 Complete | Avg Train Loss: 1.3373
📊 Validation Loss: 0.6704
Epoch 2/4 | Batch 0 | Loss: 0.8241
Epoch 2/4 |

##Evaluation — Positive@1 & MRR
Samples up to 500 validation triplets and re-ranks each query against its 1 positive
+ 4 negatives using cosine similarity. Reports:
- **Positive@1**: fraction of queries where the correct passage ranks first
- **MRR**: Mean Reciprocal Rank across all queries
- **Jurafsky-specific Positive@1**: subset breakdown for textbook retrieval

Random baseline with 5 candidates = **0.20**.

In [17]:
import random
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

model.eval()

sample = random.sample(val_triplets, min(500, len(val_triplets)))

correct_at_1 = 0
mrr_total = 0.0
jur_correct_at_1 = 0
jur_total = 0

for t in sample:
    query = t["query"]
    candidates = [t["positive"]] + t["negatives"][:4]

    with torch.no_grad():
        q_emb = model.encode([query], tokenizer)
        c_emb = model.encode(candidates, tokenizer)

    scores = cosine_similarity(q_emb, c_emb)[0]
    ranked = np.argsort(scores)[::-1]

    rank_of_positive = list(ranked).index(0) + 1

    if rank_of_positive == 1:
        correct_at_1 += 1
        if t.get("source") == "jurafsky":
            jur_correct_at_1 += 1

    if t.get("source") == "jurafsky":
        jur_total += 1

    mrr_total += 1.0 / rank_of_positive

print("Random Positive@1 baseline with 5 candidates: 0.20")
print("Positive@1 accuracy:", correct_at_1 / len(sample))
print("MRR:", mrr_total / len(sample))

if jur_total > 0:
    print("Jurafsky Positive@1 accuracy in sample:", jur_correct_at_1 / jur_total)
else:
    print("No Jurafsky examples in sampled validation subset.")


Random Positive@1 baseline with 5 candidates: 0.20
Positive@1 accuracy: 0.926
MRR: 0.9621666666666666
Jurafsky Positive@1 accuracy in sample: 0.75


##Interactive Jurafsky Search
Loads the trained model and tokenizer, encodes all clean Jurafsky passages into
a dense index, then runs an interactive search loop.

Enter any NLP-related query to retrieve the top 5 most semantically similar
passages from *Speech and Language Processing* (Jurafsky & Martin).
Type `quit` to exit.

In [ ]:
import json
import sys
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import importlib

PROJECT_ROOT = Path("/content/NLP_final_project").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.encoder as encoder_module
importlib.reload(encoder_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer

MODEL_PATH = PROJECT_ROOT / "models" / "encoder_transformer.pth"
TOKENIZER_PATH = PROJECT_ROOT / "models" / "tokenizer_vocab.json"
CHUNKS_PATH = PROJECT_ROOT / "data" / "jurafsky_chunks" / "chunks.jsonl"

DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/NLP_project/saved_models")

if not MODEL_PATH.exists() and (DRIVE_MODEL_DIR / "encoder_transformer.pth").exists():
    MODEL_PATH = DRIVE_MODEL_DIR / "encoder_transformer.pth"

if not TOKENIZER_PATH.exists() and (DRIVE_MODEL_DIR / "tokenizer_vocab.json").exists():
    TOKENIZER_PATH = DRIVE_MODEL_DIR / "tokenizer_vocab.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError("Model not found. Run training cell first.")
if not TOKENIZER_PATH.exists():
    raise FileNotFoundError("Tokenizer not found. Run training cell first.")
if not CHUNKS_PATH.exists():
    raise FileNotFoundError("Jurafsky chunks not found. Run chunk rebuild cell first.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EncoderOnlyTransformer.load(str(MODEL_PATH), device=device)
model.eval()
tokenizer = SimpleTokenizer.load(str(TOKENIZER_PATH))

def is_good_search_chunk(text):
    if not isinstance(text, str):
        return False

    text = text.strip()
    words = text.split()

    if len(words) < 80:
        return False

    alpha_chars = sum(ch.isalpha() for ch in text)
    if alpha_chars < 250:
        return False

    lower = text.lower()

    bad_markers = [
        "acknowledg",
        "bibliography",
        "references",
        "author index",
        "subject index",
        "index",
        "figure c.",
        "tag description example",
        "ucrel",
    ]

    if any(marker in lower[:500] for marker in bad_markers):
        return False

    if text.count(",") > 70 and len(words) < 260:
        return False

    return True


passages = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue

        text = data.get("text", "")

        if is_good_search_chunk(text):
            passages.append(text.strip())


if not passages:
    raise ValueError("No good Jurafsky passages loaded. Check chunking/filtering.")


batch_size = 128
embeddings = []

for i in range(0, len(passages), batch_size):
    batch = passages[i:i + batch_size]

    with torch.no_grad():
        emb = model.encode(batch, tokenizer)

    embeddings.append(emb)

    if i % (batch_size * 20) == 0:
        print(f"Encoded {i}/{len(passages)}")

passage_embeddings = np.vstack(embeddings)

print("Type 'quit' to exit.\n")

while True:
    try:
        query = input("Enter query: ")
    except EOFError:
        break

    query = query.strip()

    if query.lower() == "quit":
        print(" Exiting...")
        break

    if not query:
        continue

    with torch.no_grad():
        query_emb = model.encode([query], tokenizer)

    similarities = cosine_similarity(query_emb, passage_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:5]

    print("\nTop 5 results:")

    for rank, idx in enumerate(top_indices, start=1):
        print("\n" + "-" * 90)
        print(f"Rank {rank} | score={similarities[idx]:.4f}")
        print(passages[idx][:1400])



Model loaded from /content/NLP_final_project/models/encoder_transformer.pth
Encoded 0/663
Type 'quit' to exit.

Enter query: Attention Mechanism

Top 5 results:

------------------------------------------------------------------------------------------
Rank 1 | score=0.5296
Section 4.5. Machine Learning of Phonological Rules 117 GEN *COMPLEX FAITHC FAITHV /?ilk−hin/ [?i.lik.hin] ?ilk.hin ?i.lik.hin?il.khin ?il.hin ?ak.pid GEN *COMPLEX ?i.lik.hin FAITHC ?i.lik.hin?il.hin ?ak.pid FAITHV L L L Figure 4.19 Version #2 (‘lenient cascade’) of Karttunen’s ﬁnite-statecas- cade implementation of OT, showing a visualization of the ca ndidate popula- tions that would be passed through each FST constraint. 4.5 M ACHINE LEARNING OF PHONOLOGICAL RULES The task of a machine learning system is to automatically induce a model MACHINE LEARNING for some domain, given some data from the domain and, sometim es, other information as well. Thus a system to learn phonological rul es would be given at least a s